# 📘 The AI Engineer's LLM Workbook

**14 Chapters · 14 Google Colab Notebooks · Beginner to Production**

---

*© 2026 JAWNVION LLC — www.jawnvion.com — peter@jawnvion.com*

*Licensed for individual use. Do not redistribute.*

---

## What's Inside

| # | Chapter |
|---|---------|
| 01 | AI Fundamentals & Problem Framing |
| 02 | Data Science Toolkit (NumPy, Pandas, Matplotlib) |
| 03 | Neural Networks from Scratch |
| 04 | Transformers Architecture Deep Dive |
| 05 | HuggingFace & Pre-Trained Models |
| 06 | QLoRA Fine-Tuning |
| 07 | DPO Alignment Training |
| 08 | Retrieval-Augmented Generation (RAG) |
| 09 | Model Evaluation & Benchmarking |
| 10 | FastAPI Deployment |
| 11 | Monitoring & Observability |
| 12 | Security for AI Systems |
| 13 | Cost Optimization & Quantization |
| 14 | Capstone: End-to-End LLM Project |

---

> **How to use:** Click **Runtime → Run All** in Google Colab, or run cells one at a time.
> Each chapter builds on the last — complete them in order for best results.

---


# Chapter 11: Monitoring & Observability
**JAWNVION LLC — AI Training Workbook**

A deployed model without observability is a black box — you won't know when it
degrades, slows down, or starts producing unsafe outputs until a user complains.
This chapter instruments the Chapter 10 inference server with production-grade
monitoring.

**What you'll learn:**
- Structured JSON logging: machine-readable request/response records
- Prometheus metrics: counters, histograms, and gauges for the `/generate` endpoint
- FastAPI middleware: instrument every request without touching handler code
- Load generation: fire N concurrent requests and observe the metrics
- Log analysis: compute latency percentiles and error rates from the log file
- Production stack: AWS CloudWatch (GovCloud) and Grafana integration overview

In [ ]:
# — Cell 1: GPU Check ——————————————————————————————————
import torch, subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free',
     '--format=csv,noheader'],
    capture_output=True, text=True
)
if result.returncode == 0:
    print('✓  GPU detected:', result.stdout.strip())
else:
    print('⚠  No GPU — inference runs on CPU (monitoring concepts work either way)')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'   PyTorch : {torch.__version__}')
print(f'   Device  : {device}')

In [ ]:
# — Cell 2: Install Packages ——————————————————————————
!pip install -q "tokenizers>=0.22,<0.24"
!pip install -q -U transformers peft accelerate
!pip install -q fastapi "uvicorn[standard]<0.30" nest-asyncio httpx
!pip install -q prometheus-client python-json-logger
print('✓  Packages installed')
print('   Key additions: prometheus-client (metrics), python-json-logger (structured logs)')

In [ ]:
# — Cell 3: Imports & Configuration ———————————————————
import torch
import time
import threading
import asyncio
import nest_asyncio
import httpx
import json
import logging
import os
import statistics
from datetime import datetime, timezone
from pathlib import Path

# Prometheus
from prometheus_client import (
    Counter, Histogram, Gauge,
    generate_latest, CONTENT_TYPE_LATEST,
    CollectorRegistry,
)

# FastAPI
from fastapi import FastAPI, Request, Response
from fastapi.responses import PlainTextResponse
from pydantic import BaseModel
from typing import Optional
import uvicorn

# HuggingFace
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

nest_asyncio.apply()

# ── Constants ─────────────────────────────────────────
BASE_MODEL   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
ADAPTER_DIR  = "/content/tinyllama-qlora"    # Chapter 6 output
MERGED_DIR   = "/content/tinyllama-merged"   # Chapter 10 output (preferred)
LOG_FILE     = "/content/inference_log.jsonl"
API_PORT     = 8001                          # 8001 avoids conflict if Ch10 server still running
MAX_NEW      = 100

print('✓  Config ready')
print(f'   Log file : {LOG_FILE}')
print(f'   API port : {API_PORT}')

In [ ]:
# — Cell 4: Structured JSON Logger ————————————————————
# JSON logs are machine-readable — you can pipe them to CloudWatch Logs,
# Elasticsearch, or Splunk without a log parser.
# Each request produces one JSON line: timestamp, latency, tokens, prompt hash, etc.

from pythonjsonlogger import jsonlogger

# Clear any existing handlers so re-running the cell doesn't duplicate logs
inference_logger = logging.getLogger("inference")
inference_logger.handlers.clear()
inference_logger.setLevel(logging.INFO)

# File handler — writes JSONL (one JSON object per line)
file_handler = logging.FileHandler(LOG_FILE, mode="a")
formatter = jsonlogger.JsonFormatter(
    fmt="%(asctime)s %(name)s %(levelname)s %(message)s",
    datefmt="%Y-%m-%dT%H:%M:%S",
)
file_handler.setFormatter(formatter)
inference_logger.addHandler(file_handler)

# Console handler — pretty-print for Colab output
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter("%(asctime)s  %(message)s", "%H:%M:%S"))
inference_logger.addHandler(console_handler)

# Test it
inference_logger.info("Logger initialised", extra={
    "event": "startup",
    "model": BASE_MODEL,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
})
print(f"\n✓  Logger ready — writing to {LOG_FILE}")
print("   Format: JSON Lines (one object per request)")

In [ ]:
# — Cell 5: Prometheus Metrics ————————————————————————
# Prometheus is the industry standard for time-series metrics.
# These three metric types cover the most important LLM serving signals.

# Use a fresh registry so re-running the cell doesn't get "already registered" errors
REGISTRY = CollectorRegistry()

# ── Counters (monotonically increasing) ───────────────
REQUEST_COUNT = Counter(
    "llm_requests_total",
    "Total inference requests",
    ["status"],          # labels: status=success | error
    registry=REGISTRY,
)

TOKEN_COUNT = Counter(
    "llm_tokens_total",
    "Total tokens generated",
    registry=REGISTRY,
)

# ── Histogram (latency distribution) ──────────────────
REQUEST_LATENCY = Histogram(
    "llm_request_latency_seconds",
    "End-to-end inference latency",
    buckets=[0.5, 1.0, 2.0, 3.0, 5.0, 10.0, float("inf")],
    registry=REGISTRY,
)

TOKENS_PER_REQUEST = Histogram(
    "llm_tokens_per_request",
    "Tokens generated per request",
    buckets=[10, 25, 50, 75, 100, 150, float("inf")],
    registry=REGISTRY,
)

# ── Gauge (current point-in-time value) ───────────────
REQUESTS_IN_FLIGHT = Gauge(
    "llm_requests_in_flight",
    "Requests currently being processed",
    registry=REGISTRY,
)

print("✓  Prometheus metrics registered")
print("   Counter  : llm_requests_total{status}  |  llm_tokens_total")
print("   Histogram: llm_request_latency_seconds  |  llm_tokens_per_request")
print("   Gauge    : llm_requests_in_flight")

In [ ]:
# — Cell 6: Load Model ————————————————————————————————
# Prefer the merged checkpoint from Chapter 10, then the LoRA adapter,
# then fall back to the base model — so the chapter is self-contained.

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token

if os.path.isdir(MERGED_DIR):
    print(f"Loading merged model from {MERGED_DIR}  (Chapter 10 output)...")
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    source = "merged (Ch10)"
elif os.path.isdir(ADAPTER_DIR):
    print(f"Loading base + LoRA adapter from {ADAPTER_DIR}  (Chapter 6 output)...")
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    model = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    source = "base + LoRA adapter (Ch6)"
else:
    print(f"Loading base model (no adapter found)...")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )
    source = "base model"

model.eval()
vram = torch.cuda.memory_allocated() / 1e9 if torch.cuda.is_available() else 0
print(f"\n✓  Model ready  |  Source: {source}  |  VRAM: {vram:.2f} GB")

In [ ]:
# — Cell 7: Instrumented FastAPI App ——————————————————
# The middleware pattern is the right way to instrument a web framework:
# it wraps EVERY request without modifying individual handler functions.
# This is exactly how production APM tools (Datadog, New Relic) work.

app = FastAPI(title="TinyLlama Monitored Server", version="2.0")

# ── Middleware — wraps every request ──────────────────
@app.middleware("http")
async def metrics_middleware(request: Request, call_next):
    """
    Runs before and after every HTTP handler.
    Records: in-flight count, latency, status.
    This middleware never touches the generate() logic.
    """
    REQUESTS_IN_FLIGHT.inc()
    start = time.time()
    try:
        response = await call_next(request)
        status = "success"
        return response
    except Exception as exc:
        status = "error"
        raise exc
    finally:
        elapsed = time.time() - start
        REQUESTS_IN_FLIGHT.dec()
        # Only record metrics for the /generate endpoint
        if request.url.path == "/generate":
            REQUEST_LATENCY.observe(elapsed)
            REQUEST_COUNT.labels(status=status).inc()

# ── Endpoints ──────────────────────────────────────────
@app.get("/health")
def health():
    return {"status": "ok", "device": str(model.device)}

@app.get("/metrics", response_class=PlainTextResponse)
def metrics():
    """Prometheus scrape endpoint — Grafana and Prometheus scrape this URL."""
    return generate_latest(REGISTRY).decode("utf-8")

class GenerateRequest(BaseModel):
    prompt: str
    max_new_tokens: Optional[int] = MAX_NEW
    temperature: Optional[float] = 0.7

@app.post("/generate")
def generate_endpoint(req: GenerateRequest):
    t0 = time.time()
    inputs = tokenizer(
        req.prompt, return_tensors="pt", truncation=True, max_length=512
    ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=req.max_new_tokens,
            do_sample=True,
            temperature=req.temperature,
            top_p=0.9,
            repetition_penalty=1.3,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )
    n_new   = out.shape[1] - inputs["input_ids"].shape[1]
    text    = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=True).strip()
    elapsed = time.time() - t0

    # Update token-level metrics
    TOKEN_COUNT.inc(n_new)
    TOKENS_PER_REQUEST.observe(n_new)

    # Write structured log record
    inference_logger.info("inference_complete", extra={
        "event":            "inference_complete",
        "elapsed_sec":      round(elapsed, 3),
        "tokens_generated": n_new,
        "tokens_per_sec":   round(n_new / elapsed, 1),
        "prompt_len_chars": len(req.prompt),
        "temperature":      req.temperature,
        "status":           "success",
    })

    return {
        "response":         text,
        "tokens_generated": n_new,
        "elapsed_sec":      round(elapsed, 3),
        "tokens_per_sec":   round(n_new / elapsed, 1),
    }

print("✓  Instrumented FastAPI app defined")
print("   Endpoints: GET /health  |  GET /metrics  |  POST /generate")

In [ ]:
# — Cell 8: Launch Monitored Server ———————————————————
SERVER_READY = threading.Event()

def run_server():
    config = uvicorn.Config(app, host="0.0.0.0", port=API_PORT, log_level="warning")
    server = uvicorn.Server(config)
    SERVER_READY.set()
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    loop.run_until_complete(server.serve())

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
SERVER_READY.wait(timeout=10)
time.sleep(1.5)

resp = httpx.get(f"http://localhost:{API_PORT}/health")
print(f"✓  Server running on port {API_PORT}")
print(f"   Health  : {resp.json()}")
print(f"   Metrics : http://localhost:{API_PORT}/metrics  (Prometheus format)")

In [ ]:
# — Cell 9: Load Generation — Fire 10 Requests ————————
# Send 10 requests to populate the metrics and logs.
# We vary the prompts so the log has interesting diversity to analyse.

PROMPTS = [
    "### Instruction:\nWhat is a neural network?\n\n### Response:\n",
    "### Instruction:\nExplain gradient descent in simple terms.\n\n### Response:\n",
    "### Instruction:\nWhat is overfitting?\n\n### Response:\n",
    "### Instruction:\nDescribe the transformer architecture.\n\n### Response:\n",
    "### Instruction:\nWhat is tokenization in NLP?\n\n### Response:\n",
    "### Instruction:\nList three advantages of fine-tuning over prompt engineering.\n\n### Response:\n",
    "### Instruction:\nWhat is the difference between precision and recall?\n\n### Response:\n",
    "### Instruction:\nExplain what embeddings are.\n\n### Response:\n",
    "### Instruction:\nWhat is RLHF?\n\n### Response:\n",
    "### Instruction:\nDescribe how RAG retrieval works.\n\n### Response:\n",
]

print("Firing 10 inference requests...")
print("─" * 55)

results = []
for i, prompt in enumerate(PROMPTS):
    resp = httpx.post(
        f"http://localhost:{API_PORT}/generate",
        json={"prompt": prompt, "max_new_tokens": 80, "temperature": 0.7},
        timeout=90,
    )
    data = resp.json()
    results.append(data)
    short_prompt = prompt.split("\n")[1][:45]
    print(f"  [{i+1:02d}]  {short_prompt:<45}  {data['elapsed_sec']:.2f}s  {data['tokens_per_sec']:.0f} tok/s")

latencies = [r["elapsed_sec"] for r in results]
tps       = [r["tokens_per_sec"] for r in results]
print("─" * 55)
print(f"  Mean latency : {statistics.mean(latencies):.2f}s")
print(f"  P95 latency  : {sorted(latencies)[int(0.95 * len(latencies)) - 1]:.2f}s")
print(f"  Mean tok/s   : {statistics.mean(tps):.1f}")
print(f"\n✓  10 requests complete — metrics and logs populated")

In [ ]:
# — Cell 10: Read Prometheus Metrics ——————————————————
# In production, Prometheus scrapes /metrics every 15s and stores the time-series.
# Here we scrape it once and parse the output to show the values.

raw = httpx.get(f"http://localhost:{API_PORT}/metrics").text

print("RAW PROMETHEUS OUTPUT (first 60 lines):")
print("─" * 55)
for line in raw.split("\n")[:60]:
    if line and not line.startswith("#"):
        print(f"  {line}")

print("─" * 55)
print()

# Parse key values from the text exposition format
import re

def scrape(metric_name, raw_text):
    """Extract all label/value pairs for a metric from Prometheus text format."""
    results = {}
    for line in raw_text.split("\n"):
        if line.startswith(metric_name) and not line.startswith("#"):
            # e.g. llm_requests_total{status="success"} 10.0
            m = re.match(rf'{metric_name}(\{{[^}}]*\}})? (\S+)', line)
            if m:
                label  = m.group(1) or ""
                value  = m.group(2)
                results[label] = float(value)
    return results

total_req   = scrape("llm_requests_total", raw)
total_tok   = scrape("llm_tokens_total",   raw)
in_flight   = scrape("llm_requests_in_flight", raw)

print("PARSED METRICS SUMMARY:")
print(f"  Total requests  : {sum(total_req.values()):.0f}")
for label, val in total_req.items():
    print(f"    {label:30s} {val:.0f}")
print(f"  Total tokens    : {sum(total_tok.values()):.0f}")
print(f"  In-flight now   : {sum(in_flight.values()):.0f}")
print()
print("✓  Prometheus scrape working — these are the numbers Grafana would graph")

In [ ]:
# — Cell 11: Log Analysis — Parse JSONL ———————————————
# The JSON log file is the audit trail — useful for debugging, billing,
# and detecting model drift (responses getting shorter over time, etc.).

records = []
with open(LOG_FILE) as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
            # Filter to inference_complete events only (skip startup log)
            if obj.get("event") == "inference_complete":
                records.append(obj)
        except json.JSONDecodeError:
            pass

if not records:
    print("No inference records found yet — make sure Cell 9 ran first")
else:
    latencies = [r["elapsed_sec"]      for r in records]
    tps_list  = [r["tokens_per_sec"]   for r in records]
    tok_list  = [r["tokens_generated"] for r in records]

    print(f"LOG ANALYSIS — {len(records)} inference records")
    print("─" * 50)
    print(f"  Latency  — mean: {statistics.mean(latencies):.2f}s  "
          f"| median: {statistics.median(latencies):.2f}s  "
          f"| max: {max(latencies):.2f}s")
    print(f"  Tok/s    — mean: {statistics.mean(tps_list):.1f}  "
          f"| min: {min(tps_list):.1f}  "
          f"| max: {max(tps_list):.1f}")
    print(f"  Tokens   — mean: {statistics.mean(tok_list):.1f}  "
          f"| total: {sum(tok_list)}")
    print()

    # Show the most recent 3 log entries
    print("RECENT LOG ENTRIES (last 3):")
    for r in records[-3:]:
        ts = r.get("asctime", r.get("timestamp", "?"))
        print(f"  {ts}  |  {r['elapsed_sec']:.2f}s  |  "
              f"{r['tokens_generated']} tok  |  {r['tokens_per_sec']:.1f} tok/s")
    print()
    print("✓  Log analysis complete")
    print(f"   Log file: {LOG_FILE}  ({os.path.getsize(LOG_FILE)} bytes)")

In [ ]:
# — Cell 12: Production Monitoring Stack ——————————————
# Nothing to execute — reference for GovCloud / on-prem deployment.

STACK_OVERVIEW = """
PRODUCTION MONITORING STACK — DocuMind Gov Enclave (AWS GovCloud)
═══════════════════════════════════════════════════════════════════

LAYER 1 — Structured Logs  (this chapter: python-json-logger)
  Colab demo  : JSON Lines written to /content/inference_log.jsonl
  Production  : JSON logs → CloudWatch Logs (GovCloud)
                           → CloudWatch Log Insights for ad-hoc queries
  What to log : timestamp, latency, token count, prompt hash (NOT the full
                prompt if it contains CUI), model version, status

LAYER 2 — Metrics  (this chapter: prometheus_client + /metrics endpoint)
  Colab demo  : GET /metrics → Prometheus text format
  Production  : Prometheus server scrapes /metrics every 15s
                Grafana reads Prometheus → live dashboards
                Alternatively: CloudWatch EMF (Embedded Metric Format) to
                emit Prometheus-style metrics natively into CloudWatch Metrics
  Key metrics : llm_requests_total  |  llm_request_latency_seconds (p50/p95/p99)
                llm_tokens_total    |  llm_requests_in_flight

LAYER 3 — Alerts  (not demoed in Colab)
  Tool        : Prometheus Alertmanager → PagerDuty / SNS
                CloudWatch Alarms → SNS → email/phone
  Example rules:
    - p95 latency > 5s for 3 consecutive minutes → page on-call
    - error rate > 1% over 5 minutes             → page on-call
    - requests_in_flight > 10 for 2 minutes      → scale-out trigger

LAYER 4 — Model Quality / Drift  (beyond this chapter)
  Risk : model outputs may degrade over time as the document corpus changes
  Tool : Periodic eval runs (Chapter 9 pipeline on a schedule) comparing
         ROUGE / BLEU vs. a fixed baseline; alert if score drops > 5%
  In GovCloud: Lambda + EventBridge to schedule nightly eval jobs

LAYER 5 — Cost & Capacity
  GPU util (nvidia-smi → CloudWatch custom metric)
  Token throughput cost model: $/1k tokens served × daily volume
  Scale trigger: when p95 latency > SLA → add GPU instance or switch to vLLM

GOVCLOUD SPECIFICS
  • CloudWatch Logs: FedRAMP-authorized, CUI-safe — use as log destination
  • CloudWatch Metrics: native dashboards without a separate Prometheus/Grafana
  • AWS X-Ray: distributed tracing for multi-service inference pipelines
  • CloudTrail: audit log of every API call to the enclave — required for CMMC
"""

print(STACK_OVERVIEW)
print("✓  Chapter 11 complete")

## Chapter 11 Complete ✓

**What happened:**
- Set up a `python-json-logger` structured logger writing JSON Lines to `/content/inference_log.jsonl`
- Defined five Prometheus metrics: two Counters, two Histograms, one Gauge
- Added a FastAPI middleware layer that instruments every request without touching handler code
- Fired 10 varied inference requests to populate metrics and logs
- Scraped `/metrics` and parsed the Prometheus text exposition format
- Analysed the JSONL log file to compute latency percentiles and throughput stats
- Mapped the full production monitoring stack for the DocuMind Gov enclave (CloudWatch)

**The three monitoring layers every LLM deployment needs:**
| Layer | What it answers | Tool (Colab) | Tool (GovCloud) |
|---|---|---|---|
| Structured logs | What happened on each request? | python-json-logger + JSONL | CloudWatch Logs |
| Metrics | How is the system performing right now? | prometheus_client + Grafana | CloudWatch Metrics / EMF |
| Alerts | When is something wrong? | Alertmanager | CloudWatch Alarms → SNS |

**Key patterns:**
- Middleware instrumentation keeps observability code out of business logic
- JSON logs are schema-on-read — add fields without breaking existing queries
- Histograms (not averages) are required for latency — p95 catches tail latency that mean hides
- Log the prompt *hash*, not the prompt text, when prompts may contain CUI

**Model drift detection** (Chapter 9 + Chapter 11 combined):
Run the Chapter 9 evaluation pipeline on a cron schedule; emit ROUGE/BLEU scores as CloudWatch custom metrics; set an alarm if the score drops more than 5% from the baseline recorded at deployment.

**Next: Chapter 12 — Security & Responsible AI**